# API

Use ```get_arxiv_search_results(search_term)``` or ```get_pubmed_search_results(search_term)``` to get ```ArticleList```.

Use ```ArticleList.get_list_of(field)``` to get list of those fields.

## Common

#### Imports

In [10]:
import xml.etree.ElementTree as ET
import requests
import json
import datetime
import re
import pandas as pd

#### API URLs

In [11]:
API_PUBMED_PMCID = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
API_PUBMED_SUMMARY = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
API_ARXIV = 'https://export.arxiv.org/api/query'

#### Article class

In [12]:
class ArticleSummary(dict):
    def __init__(self, doi: str, id: str, title: str, pubdate: str, journal: str, authors: list[str], keywords: list[str], abstract: str):
        super().__init__(
            doi=doi,
            id=id,
            title=title,
            pubdate=pubdate,
            journal=journal,
            authors=authors,
            keywords=keywords,
            abstract=abstract
        )
        self.doi = doi
        self.id = id
        self.title = title
        self.pubdate = pubdate
        self.journal = journal
        self.authors = authors
        self.keywords = keywords
        self.abstract = abstract

#### Article List

In [13]:
class ArticleField:
    DOI = 'doi'
    ID = 'id'
    TITLE = 'title'
    PUBDATE = 'pubdate'
    JOURNAL = 'journal'
    AUTHORS = 'authors'
    KEYWORDS = 'keywords'
    ABSTRACT = 'abstract'

class ArticleList(list):
    def __init__(self, articles: list[ArticleSummary]):
        super().__init__(articles)
    
    def get_list_of(self, field: ArticleField) -> list:
        return [getattr(article, field) for article in self]

#### Requester

In [14]:
def request_with_retries(url, params, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()
            return response
        except requests.RequestException as e:
            print(f'Attempt {attempt + 1} failed: {e}')
            if attempt == max_retries - 1:
                raise
    return None

## Searchers

### PubMed

In [15]:
def get_pmcids(search_term: str, max_results: int = 100) -> ArticleList:
    response = request_with_retries(
        url = API_PUBMED_PMCID,
        params = {
            'db': 'pubmed',
            'term': search_term,
            'retmax': max_results,
            'sort': 'relevance'
        }
    )

    if response is None:
        print('Failed to retrieve data after multiple attempts.')
        return []

    root = ET.fromstring(response.text)
    pmcids = root.findall('.//IdList/Id')

    return [pmcid.text for pmcid in pmcids]

def get_pubmed_search_results(search_term: str, max_results: int = 100) -> list[ArticleSummary]:
    pmcids = get_pmcids(search_term, max_results)
    response = request_with_retries(
        url = API_PUBMED_SUMMARY,
        params = {
            'id': ','.join(pmcids),
            'retmode': 'xml',
            'db': 'pubmed'
        }
    )

    if response is None:
        print('Failed to retrieve data after multiple attempts.')
        return []

    root = ET.fromstring(response.text)
    articles = root.findall('.//PubmedArticle')
    summaries = []
    for a in articles:
        doi = a.findtext('.//ArticleIdList/ArticleId[@IdType="doi"]') or ''
        id = a.findtext('.//MedlineCitation/PMID') or ''
        title = a.findtext('.//Article/ArticleTitle') or ''
        pubdate = a.findtext('.//Article/Journal/JournalIssue/PubDate/Year') or ''
        journal = a.findtext('.//Article/Journal/Title') or ''
        authors = [(author.findtext('LastName') or '') + ' ' + (author.findtext('ForeName') or '') for author in a.findall('.//Article/AuthorList/Author')]
        keywords = [kw.text for kw in a.findall('.//KeywordList/Keyword')]
        abstract = a.findtext('.//Article/Abstract/AbstractText') or ''
        summary = ArticleSummary(doi, id, title, pubdate, journal, authors, keywords, abstract)
        summaries.append(summary)

    return ArticleList(summaries)

In [16]:
pubmed_article_list = get_pubmed_search_results('cancer cells')
print(pubmed_article_list.get_list_of(ArticleField.TITLE))

json_summaries = json.dumps(pubmed_article_list)
with open('article_search_results_pubmed.json', 'w') as f:
    f.write(json_summaries)

df_pubmed = pd.DataFrame(pubmed_article_list)
df_pubmed

['Persistent Cancer Cells: The Deadly Survivors.', 'Polyploid giant cancer cells and cancer progression.', 'Cancer cells resist hyperthermia due to its obstructed activation of caspase 3.', 'Interconversion of Cancer Cells and Induced Pluripotent Stem Cells.', 'Cancer cells as a new source of induced pluripotent stem cells.', 'Infection of non-cancer cells: A barrier or support for oncolytic virotherapy?', 'Free cancer cells in gastric cancer - methods of detection, clinical and prognostic importance (meta-analysis).', 'The bacteria inside human cancer cells: Mainly as cancer promoters.', 'Effects of local anesthetics on cancer cells.', 'Lipid Metabolic Regulatory Crosstalk Between Cancer Cells and Tumor-Associated Macrophages.', 'Water Dynamics in Cancer Cells: Lessons from Quasielastic Neutron Scattering.', 'Are cancer cells really softer than normal cells?', 'Polyploid giant cancer cells: origin, possible pathways of formation, characteristics, and mechanisms of regulation.', 'Repro

,doi,id,title,pubdate,journal,authors,keywords,abstract
0,10.1016/j.cell.2020.10.027,33186528,Persistent Cancer Cells: The Deadly Survivors.,2020,Cell,"[Shen Shensi, Vagner Stéphan, Robert Caroline]","[adaptive resistance, cancer drug addiction, c...",Persistent cancer cells are the discrete and u...
1,10.3389/fcell.2022.1017588,36274852,Polyploid giant cancer cells and cancer progre...,2022,Frontiers in cell and developmental biology,"[Zhou Xinyue, Zhou Mingming, Zheng Minying, Ti...","[cancer stem cells, chemoradiotherapy resistan...",Polyploid giant cancer cells
2,10.1016/j.rpor.2020.02.008,32194353,Cancer cells resist hyperthermia due to its ob...,2020,Reports of practical oncology and radiotherapy...,"[Tang Xiaoren, Cao Feng, Ma Weiyuan, Tang Yini...",[Cancer cells resist hyperthermia],It is well known that inducing hyperthermia is...
3,10.3390/cells13020125,38247819,Interconversion of Cancer Cells and Induced Pl...,2024,Cells,"[Sarker Drishty B, Xue Yu, Mahmud Faiza, Jocel...","[cancer stem cell models, cancer stem cells, i...","Cancer cells, especially cancer stem cells (CS..."
4,10.1186/s13287-022-03145-y,36064437,Cancer cells as a new source of induced plurip...,2022,Stem cell research & therapy,"[Shamsian Azam, Sahebnasagh Roxana, Norouzy Am...","[Cancer cell reprogramming, Cancer-iPSCs, Indu...","Over the last 2 decades, induced pluripotent s..."
...,...,...,...,...,...,...,...,...
95,10.3390/cancers12102886,33050036,Increased Expression of NPM1 Suppresses p27,2020,Cancers,"[Kometani Tatsuya, Arai Takuya, Chibazakura Taku]","[CDK inhibitor, cancer cells, cell proliferati...",p27
96,10.1007/s11302-022-09854-6,35235139,P2X7 receptor involved in antitumor activity o...,2023,Purinergic signalling,"[Han Yue, Bai Can, He Xi-Meng, Ren Qing-Ling]","[Atractylenolide I, Hela cells, Human cervical...",Atractylenolide I (Atr-I) was found to sensiti...
97,10.1007/s13577-019-00275-z,31441010,Interaction between cancer cells and cancer-as...,2019,Human cell,"[Hisamitsu Shoshi, Miyashita Tomoyuki, Hashimo...","[Cancer-associated fibroblasts, Cisplatin, Lun...",Regrowth of cancer cells following chemotherap...
98,10.1016/j.acthis.2022.151938,35981451,"RILP inhibits proliferation, migration, and in...",2022,Acta histochemica,"[Wang Zhen, Zhou Yunhe, Nie Dongsong, Tan Yan,...","[Lysosome, PC3 prostate cancer cells, RILP, Ra...",RILP (Rab-interacting lysosomal protein) is a ...


### ArXiv

In [17]:
def get_arxiv_search_results(search_term: str, max_results: int = 100) -> ArticleList:
    response = request_with_retries(
        url = API_ARXIV,
        params = {
            'search_query': search_term,
            'start': 0,
            'max_results': max_results
        }
    )

    if response is None:
        print('Failed to retrieve data after multiple attempts.')
        return []

    root = ET.fromstring(response.text)
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
        "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
    }
    entries = root.findall('atom:entry', ns)
    summaries = []
    for entry in entries:
        id = entry.findtext('atom:id', namespaces=ns)
        doi = entry.findtext('arxiv:doi', namespaces=ns)
        title = entry.findtext('atom:title', namespaces=ns)
        pubdate = datetime.datetime.strptime(entry.findtext('atom:published', namespaces=ns), '%Y-%m-%dT%H:%M:%S%z').strftime('%Y')
        journal = 'arXiv'
        authors = [author.findtext('atom:name', namespaces=ns) for author in entry.findall('atom:author', namespaces=ns)]
        keywords_text = entry.findtext('arxiv:comment', namespaces=ns)
        keywords_split = re.split(r'keywords:\s*', keywords_text, flags=re.IGNORECASE) if keywords_text else []
        if len(keywords_split) > 1:
            keywords = [kw.strip() for kw in keywords_split[1].split(',')]
        else:
            keywords = []
        abstract_element = entry.find('atom:summary', namespaces=ns)
        abstract = ''.join(abstract_element.itertext()).strip() if abstract_element is not None else ''
        summary = ArticleSummary(doi, id, title, pubdate, journal, authors, keywords, abstract)
        summaries.append(summary)

    return ArticleList(summaries)

In [18]:
arxiv_article_list = get_arxiv_search_results('UAV')
print(arxiv_article_list.get_list_of(ArticleField.TITLE))

json_arxiv_summaries = json.dumps(arxiv_article_list)
with open('article_search_results_arxiv.json', 'w') as f:
    f.write(json_arxiv_summaries)

df_arxiv = pd.DataFrame(arxiv_article_list)
df_arxiv

['UAV-VLRR: Vision-Language Informed NMPC for Rapid Response in UAV Search and Rescue', 'Capacity Characterization of UAV-Enabled Two-User Broadcast Channel', 'Repeatedly Energy-Efficient and Fair Service Coverage: UAV Slicing (Proactive UAV Network Slicing for URLLC and Mobile Broadband Service Multiplexing)', 'Common Throughput Maximization in UAV-Enabled OFDMA Systems with Delay Consideration', 'Hybrid Offline-Online Design for UAV-Enabled Data Harvesting in Probabilistic LoS Channel', 'Trajectory and Transmit Power Optimization for IRS-Assisted UAV Communication under Malicious Jamming', 'IRS-Aided Energy Efficient UAV Communication', 'Exploiting NOMA for Multi-Beam UAV Communication in Cellular Uplink', 'Robust Trajectory and Communication Design in IRS-Assisted UAV Communication under Malicious Jamming', 'The Design of Autonomous UAV Prototypes for Inspecting Tunnel Construction Environment', 'RIS-assisted UAV Communications for IoT with Wireless Power Transfer Using Deep Reinfor

,doi,id,title,pubdate,journal,authors,keywords,abstract
0,NaN,http://arxiv.org/abs/2503.02465v2,UAV-VLRR: Vision-Language Informed NMPC for Ra...,2025,arXiv,"[Yasheerah Yaqoot, Muhammad Ahsan Mustafa, Ole...",[],Emergency search and rescue (SAR) operations o...
1,NaN,http://arxiv.org/abs/1801.00443v2,Capacity Characterization of UAV-Enabled Two-U...,2018,arXiv,"[Qingqing Wu, Jie Xu, Rui Zhang]",[],Although prior works have exploited the UAV's ...
2,NaN,http://arxiv.org/abs/1912.03600v5,Repeatedly Energy-Efficient and Fair Service C...,2019,arXiv,"[Peng Yang, Xing Xi, Tony Q. S. Quek, Jingxuan...",[],Unmanned aerial vehicle (UAV) networks are con...
3,NaN,http://arxiv.org/abs/1801.00444v2,Common Throughput Maximization in UAV-Enabled ...,2018,arXiv,"[Qingqing Wu, Rui Zhang]",[],The use of unmanned aerial vehicles (UAVs) as ...
4,NaN,http://arxiv.org/abs/1907.06181v2,Hybrid Offline-Online Design for UAV-Enabled D...,2019,arXiv,"[Changsheng You, Rui Zhang]",[],This paper considers an unmanned aerial vehicl...
...,...,...,...,...,...,...,...,...
95,NaN,http://arxiv.org/abs/1906.07346v1,Energy Efficiency Maximization for Full-Duplex...,2019,arXiv,"[Bin Duo, Qingqing Wu, Xiaojun Yuan, Rui Zhang]",[],This letter proposes a new full-duplex (FD) se...
96,NaN,http://arxiv.org/abs/2002.00146v1,Sensing and Communication Tradeoff Design for ...,2020,arXiv,"[Shuhang Zhang, Hongliang Zhang, Lingyang Song...",[],"In this paper, we consider the cellular Intern..."
97,NaN,http://arxiv.org/abs/2207.01768v1,Rank-Based Filter Pruning for Real-Time UAV Tr...,2022,arXiv,"[Xucheng Wang, Dan Zeng, Qijun Zhao, Shuiwang Li]",[],Unmanned aerial vehicle (UAV) tracking has wid...
98,NaN,http://arxiv.org/abs/2409.03245v1,UAV (Unmanned Aerial Vehicles): Diverse Applic...,2024,arXiv,"[Md. Mahfuzur Rahman, Sunzida Siddique, Marufa...",[],"Unmanned Aerial Vehicles (UAVs), have greatly ..."
